In [34]:
d_model = 64        # hidden size
history_len = 30
forecast_len = 1


GRN (Gated Residual Network)

Encounters: Inside the VSN.
What it does: Smart non-linear processing.

Action: It's a building block. It takes an input vector and:
Tries to process it with dense layers + ELU (non-linearity).
Uses a Gate (sigmoid) to decide: "Should I use this processed version, or just keep the original input?"

Why: It allows the network to learn complex patterns when needed, or just pass simple signals through untouched.

In [35]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class GRN(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()

        self.fc1 = nn.Linear(input_dim, output_dim)
        self.fc2 = nn.Linear(output_dim, output_dim)

        self.gate = nn.Linear(output_dim, output_dim)
        self.layer_norm = nn.LayerNorm(output_dim)

        self.project_residual = (
            nn.Linear(input_dim, output_dim)
            if input_dim != output_dim else None
        )

    def forward(self, x):
        residual = x if self.project_residual is None else self.project_residual(x)

        x = F.elu(self.fc1(x))
        x = self.fc2(x)

        gate = torch.sigmoid(self.gate(x))
        x = gate * x + (1 - gate) * residual

        return self.layer_norm(x)

VariableSelectionNetwork (VSN)
Called Third: self.past_vsn(...) and self.future_vsn(...)

Role: The Smart Filter.

Action: It looks at the separate feature embeddings from the FeatureEncoder and asks: "Which of these features are actually useful right now?"
It uses a Weight Generator (a sub-network) to assign an importance score (e.g., Price: 0.8, Temp: 0.1, Noise: 0.1).
It creates a weighted combination of the features.

Result: A single, clean vector for each time step containing only the relevant information.
Sub-class used: GRN (Gated Residual Network)
Used by the VSN to process features and generate weights.
It allows the network to apply non-linear processing (using ELU activation) or skip processing entirely (using a Gate), giving it flexibility for both simple and complex patterns.

In [36]:
class VariableSelectionNetwork(nn.Module):
    def __init__(self, num_features, d_model):
        super().__init__()
        self.num_features = num_features

        self.feature_grns = nn.ModuleList(
            [GRN(d_model, d_model) for _ in range(num_features)]
        )

        self.weight_grn = GRN(d_model, d_model)
        self.weight_layer = nn.Linear(d_model, num_features)

    def forward(self, x):
        """
        x: [B, T, F, D]
        """
        B, T, F, D = x.shape

        # Transform each feature separately
        transformed = []
        for i in range(F):
            feat = x[:, :, i, :]  # [B, T, D]
            transformed.append(self.feature_grns[i](feat))

        transformed = torch.stack(transformed, dim=2)  # [B, T, F, D]

        # Compute weights
        context = transformed.mean(dim=2)  # [B, T, D]
        weight_context = self.weight_grn(context)
        weights = self.weight_layer(weight_context)  # [B, T, F]
        weights = torch.softmax(weights, dim=-1)

        # Weighted sum
        output = (transformed * weights.unsqueeze(-1)).sum(dim=2)

        return output

StaticEncoder
Called First: self.static_encoder(static)

Role: Metadata Embedder.

Action: It takes your unchanging "static" data (like Store ID or Item ID) and transforms it into a feature vector (d_model).
Why: This gives the model "context." The model now knows what it is predicting for, before it even looks at the time-series history.

In [37]:
class StaticEncoder(nn.Module):
    def __init__(self, input_dim, d_model):
        super().__init__()
        self.encoder = nn.Linear(input_dim, d_model)

    def forward(self, x):
        return self.encoder(x)


FeatureEncoder 

What it does: Independent projecting.

Action: It takes your time-series data (e.g., 4 features like Price, Temp, Sales). It has 4 separate tiny distinct neural networks (Linear layers) inside it.
Feature 1 goes through Network 1.
Feature 2 goes through Network 2.
...

Output: [Batch, Time, 4, 64] — Each feature is now a rich vector of size 64, but they are still separate.

In [38]:
class FeatureEncoder(nn.Module):
    def __init__(self, num_features, d_model):
        super().__init__()
        self.encoders = nn.ModuleList([
            nn.Linear(1, d_model) for _ in range(num_features)
        ])

    def forward(self, x):
        """
        x: [B, T, F]
        returns: [B, T, F, d_model]
        """
        outs = []
        for i, enc in enumerate(self.encoders):
            outs.append(enc(x[:, :, i:i+1]))
        return torch.stack(outs, dim=2)


In [39]:
import torch.nn as nn


class TemporalAttention(nn.Module):
    """
    Multi-head self-attention across the time dimension.

    Receives the concatenated past + future VSN output of shape [B, T, D],
    lets every time step attend to every other time step, then applies a
    residual connection and LayerNorm for training stability.
    """

    def __init__(self, d_model: int, num_heads: int = 4):
        super().__init__()
        # batch_first=True so tensors are [B, T, D] throughout
        self.attn = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.layer_norm = nn.LayerNorm(d_model)

    def forward(self, x):
        """
        Args:
            x: [B, T, D]  –  output from the VSN stage
        Returns:
            [B, T, D]     –  context-enriched representation
        """
        attn_out, _ = self.attn(x, x, x)       # self-attention (Q = K = V = x)
        return self.layer_norm(x + attn_out)    # residual + norm


MinimalTFT (The Conductor)

Role: This is the main class that orchestrates the entire process.

Action: It accepts the raw data (static, past, future) and passes it stage-by-stage through the other components. It manages the flow from encoding $\to$ variable selection $\to$ temporal attention $\to$ prediction.

In [40]:
class MinimalTFT(nn.Module):
    def __init__(self,
                 static_dim,
                 past_dim,
                 future_dim,
                 d_model=64,
                 num_heads=4):
        super().__init__()

        # Encoders
        self.static_encoder = StaticEncoder(static_dim, d_model)
        self.past_encoder = FeatureEncoder(past_dim, d_model)
        self.future_encoder = FeatureEncoder(future_dim, d_model)

        # Variable selection
        self.past_vsn = VariableSelectionNetwork(past_dim, d_model)
        self.future_vsn = VariableSelectionNetwork(future_dim, d_model)

        # Attention
        self.temporal_attn = TemporalAttention(d_model, num_heads)

        # Output head
        self.output_layer = nn.Linear(d_model, 1)

        

    def forward(self, static, past, future):
        """
        static: [B, S]
        past:   [B, T_hist, O]
        future: [B, T_hist + T_fut, K]
        """

        B, T_hist, O = past.shape
        T_total = future.shape[1]

        # Encode
        static_emb = self.static_encoder(static)

        past_emb = self.past_encoder(past)
        future_emb = self.future_encoder(future)


        past_sel = self.past_vsn(past_emb)
        future_sel = self.future_vsn(future_emb)

        # Combine timeline
        temporal_input = torch.cat([past_sel, future_sel[:, T_hist:]], dim=1)

        # Attention
        attn_out = self.temporal_attn(temporal_input)

        # Forecast
        forecast = self.output_layer(attn_out[:, -forecast_len:])
        return forecast.squeeze(-1)


In [41]:
B = 4
static = torch.randn(B, 5)
past = torch.randn(B, 30, 1)
future = torch.randn(B, 37, 4)

model = MinimalTFT(5, 1, 4)
out = model(static, past, future)

print(out.shape)  # [B, 7]


torch.Size([4, 1])


Raw Inputs
[B, T, F]
    ->
FeatureEncoder
[B, T, F, D]
    ->
VariableSelectionNetwork
[B, T, D]
    ->
TemporalAttention
[B, T, D]
    ->
Output Head
[B, T_fut]


### Synthetic data generator to check basline


In [42]:
import torch
import numpy as np

def generate_batch(batch_size=32, history=30, forecast=7):
    t_total = history + forecast
    
    base = torch.randn(batch_size, 1) * 5
    
    time = torch.arange(t_total).float()
    season = torch.sin(2 * np.pi * time / 7)

    sales = base + season + 0.1 * torch.randn(batch_size, t_total)
    
    past = sales[:, :history].unsqueeze(-1)
    future_target = sales[:, history:]
    
    # known future feature: day-of-week sine signal
    known = season.unsqueeze(0).repeat(batch_size, 1)
    known = known.unsqueeze(-1)
    
    static = torch.randn(batch_size, 5)
    
    return static, past, known, future_target

In [43]:
model = MinimalTFT(5, 1, 1)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.L1Loss()

for step in range(500):
    static, past, known, target = generate_batch()

    optimizer.zero_grad()
    output = model(static, past, known)
    loss = criterion(output, target)

    loss.backward()
    optimizer.step()

    if step % 50 == 0:
        print(step, loss.item())

0 4.047966480255127
50 1.1132538318634033
100 0.6565336585044861
150 0.6484569907188416
200 0.6630029082298279
250 0.6498713493347168
300 0.663101315498352
350 0.633732795715332
400 0.6737531423568726
450 0.6475951075553894


### Compare current TFT with LSTM(knows only past no known feature data)

In [44]:
import torch
import torch.nn as nn

class LSTMForecast(nn.Module):
    def __init__(self, input_dim=1, hidden_dim=64, forecast_len=7):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, forecast_len)

    def forward(self, past):
        """
        past: [B, T_hist, 1]
        """
        _, (h, _) = self.lstm(past)
        h = h.squeeze(0)          # [B, hidden_dim]
        return self.fc(h)         # [B, forecast_len]

In [45]:
tft_model = MinimalTFT(5, 1, 1)
tft_optimizer = torch.optim.Adam(tft_model.parameters(), lr=1e-3)
criterion = nn.L1Loss()

for step in range(500):
    static, past, known, target = generate_batch()

    tft_optimizer.zero_grad()
    output = tft_model(static, past, known)
    loss = criterion(output, target)

    loss.backward()
    tft_optimizer.step()

    if step % 100 == 0:
        print("TFT", step, loss.item())

TFT 0 3.512704372406006
TFT 100 0.7507451176643372
TFT 200 0.6521508097648621
TFT 300 0.6509332656860352
TFT 400 0.6523069143295288


In [46]:
lstm_model = LSTMForecast()
lstm_optimizer = torch.optim.Adam(lstm_model.parameters(), lr=1e-3)

for step in range(500):
    _, past, _, target = generate_batch()

    lstm_optimizer.zero_grad()
    output = lstm_model(past)
    loss = criterion(output, target)

    loss.backward()
    lstm_optimizer.step()

    if step % 100 == 0:
        print("LSTM", step, loss.item())

LSTM 0 4.111141204833984
LSTM 100 1.0445259809494019
LSTM 200 0.32837650179862976
LSTM 300 0.2205585539340973
LSTM 400 0.10584306716918945


TFT converges early, but at 400 steps they almost hav similar loss because the synthetic data is simple even LSTM can learn periodic patterns easily. So attention dont dominate here.


Upgrade the Synthetic Data

Modify generator to include:

1️⃣ Regime change

Seasonality shifts mid-series.

2️⃣ Promotion spikes (known future feature)

Add binary spike days.

3️⃣ Static-based variation

Make different static inputs change amplitude.

This is where:

Variable selection

Static context

Known future features

start to matter.

Why This Matters

Right now:

LSTM can memorize simple periodicity.

But if:

Regime changes

External known features exist

Different entities behave differently

Then:

TFT should become clearly superior.

In [47]:
import torch
import numpy as np

def generate_batch_v2(batch_size=32, history=30, forecast=7):
    T_total = history + forecast
    
    # ---------- STATIC ----------
    static = torch.randn(batch_size, 5)
    amplitude = 5 + static[:, 0:1] * 2  # [B,1]
    
    # ---------- TIME ----------
    t = torch.arange(T_total).float()
    
    weekly = torch.sin(2 * np.pi * t / 7)          # [T]
    monthly = torch.sin(2 * np.pi * t / 30)        # [T]
    
    regime = torch.where(
        t > T_total // 2,
        torch.tensor(1.5),
        torch.tensor(1.0)
    )  # [T]
    
    # ---------- PROMO ----------
    promo = torch.zeros(batch_size, T_total)
    for b in range(batch_size):
        promo_days = torch.randint(0, T_total, (3,))
        promo[b, promo_days] = 1.0
    
    # ---------- SALES ----------
    # Broadcasting works automatically
    base = amplitude * weekly  # [B, T]
    base += 0.5 * monthly      # monthly broadcasts
    
    sales = base * regime      # regime broadcasts
    sales += 8 * promo
    sales += 0.5 * torch.randn(batch_size, T_total)
    
    # ---------- SPLIT ----------
    past = sales[:, :history].unsqueeze(-1)
    target = sales[:, history:]
    
    known = torch.stack([
        weekly.unsqueeze(0).repeat(batch_size, 1),
        monthly.unsqueeze(0).repeat(batch_size, 1),
        promo
    ], dim=-1)
    
    return static, past, known, target

In [48]:
tft_model = MinimalTFT(5, 1, 3)
tft_optimizer = torch.optim.Adam(tft_model.parameters(), lr=1e-3)
criterion = nn.L1Loss()

for step in range(500):
    static, past, known, target = generate_batch_v2()

    tft_optimizer.zero_grad()
    output = tft_model(static, past, known)
    loss = criterion(output, target)

    loss.backward()
    tft_optimizer.step()

    if step % 100 == 0:
        print("TFT", step, loss.item())

TFT 0 5.002617835998535
TFT 100 4.960803508758545
TFT 200 5.003841876983643
TFT 300 5.337942600250244
TFT 400 4.524849891662598


In [49]:
lstm_model = LSTMForecast()
lstm_optimizer = torch.optim.Adam(lstm_model.parameters(), lr=1e-3)

for step in range(500):
    _, past, _, target = generate_batch_v2()

    lstm_optimizer.zero_grad()
    output = lstm_model(past)
    loss = criterion(output, target)

    loss.backward()
    lstm_optimizer.step()

    if step % 100 == 0:
        print("LSTM", step, loss.item())

LSTM 0 5.083031177520752
LSTM 100 2.2005913257598877
LSTM 200 1.702679991722107
LSTM 300 1.362618327140808
LSTM 400 1.5472126007080078


Why TFT wins here

Your new synthetic includes:

✅ Static-dependent amplitude

TFT sees static → scales properly
LSTM doesn’t use static at all.

✅ Promotion spikes (known future feature)

TFT sees promo directly in future window.
LSTM must guess spikes from past.

That’s almost impossible.

✅ Regime change

Attention helps detect:

time shift

pattern boundary

LSTM struggles with longer shifts.

✅ Multi-frequency seasonality

TFT can:

weigh weekly vs monthly differently

LSTM has to encode everything in hidden state.

In [50]:
class MinimalTFT(nn.Module):
    def __init__(self,
                 num_items,
                 num_stores,
                 num_cats,
                 num_states,
                 past_dim,
                 future_dim,
                 forecast_len=7,
                 d_model=64,
                 num_heads=4):
        super().__init__()

        self.forecast_len = forecast_len
        self.d_model = d_model

        # -------- Static Embeddings --------
        self.item_emb = nn.Embedding(num_items, d_model)
        self.store_emb = nn.Embedding(num_stores, d_model)
        self.cat_emb = nn.Embedding(num_cats, d_model)
        self.state_emb = nn.Embedding(num_states, d_model)

        self.static_proj = nn.Linear(d_model * 4, d_model)

        # -------- Encoders --------
        self.past_encoder = FeatureEncoder(past_dim, d_model)
        self.future_encoder = FeatureEncoder(future_dim, d_model)

        # -------- Variable Selection --------
        self.past_vsn = VariableSelectionNetwork(past_dim, d_model)
        self.future_vsn = VariableSelectionNetwork(future_dim, d_model)

        # -------- Temporal Attention --------
        self.temporal_attn1 = TemporalAttention(d_model, num_heads)
        self.dropout = nn.Dropout(p=0.1)
        self.temporal_attn2 = TemporalAttention(d_model, num_heads)

        # -------- Dropout --------
        self.dropout = nn.Dropout(0.1)
        
        # -------- Output Head --------
        self.output_layer = nn.Linear(d_model, 1)

        self.layernorm = nn.LayerNorm(d_model)

    def forward(self, static, past, future):
        """
        static: [B, 4]  (Long)
        past:   [B, T_hist, past_dim]
        future: [B, T_hist + T_fut, future_dim]
        """

        B, T_hist, _ = past.shape
        T_total = future.shape[1]

        # -------- Static Embedding --------
        item_vec = self.item_emb(static[:, 0])
        store_vec = self.store_emb(static[:, 1])
        cat_vec = self.cat_emb(static[:, 2])
        state_vec = self.state_emb(static[:, 3])

        static_concat = torch.cat(
            [item_vec, store_vec, cat_vec, state_vec],
            dim=-1
        )

        static_emb = self.static_proj(static_concat)  # [B, d_model]

        # -------- Encode Temporal Features --------
        past_emb = self.past_encoder(past)
        future_emb = self.future_encoder(future)

        past_sel = self.past_vsn(past_emb)
        future_sel = self.future_vsn(future_emb)



        # Combine past + future forecast window
        temporal_input = torch.cat(
            [past_sel, future_sel[:, T_hist:]],
            dim=1
        )

        # -------- Inject Static Context --------
        static_expanded = static_emb.unsqueeze(1).expand(
            -1,
            temporal_input.size(1),
            -1
        )

        temporal_input = temporal_input + static_expanded

        temporal_input = self.layernorm(temporal_input)

        # -------- Attention --------
        attn_out = self.temporal_attn1(temporal_input)
        attn_out = self.dropout(attn_out)
        attn_out = self.temporal_attn2(attn_out)
        
        # -------- Forecast --------
        forecast = self.output_layer(attn_out[:, -self.forecast_len:])
        return forecast.squeeze(-1)

        return forecast.squeeze(-1)

In [51]:
import pandas as pd
import numpy as np

sales = pd.read_csv("sales_train_validation.csv")
calendar = pd.read_csv("calendar.csv")

In [52]:
num_items = sales["item_id"].nunique()
num_stores = sales["store_id"].nunique()
num_cats = sales["cat_id"].nunique()
num_states = sales["state_id"].nunique()
num_items,num_stores,num_cats,num_states

(3049, 10, 3, 3)

In [53]:
# Select first 200 series only
sales = sales.iloc[:200]

We select:
- A subset of series (e.g., first 200).
- The last 400 days of sales data.

This keeps the dataset computationally manageable while preserving recent temporal patterns.

In [54]:
sales_values = sales.iloc[:, -400:].values   # shape: [200, 400]

In [55]:
calendar_subset = calendar.iloc[-400:]

In [56]:
known_features = calendar_subset[[
    "wday",
    "month",
    "snap_CA",
    "snap_TX",
    "snap_WI"
]].values

From the calendar dataset, we extract known future features such as:

- `wday` (weekday index)
- `month`
- `snap_CA`, `snap_TX`, `snap_WI`

These features are known ahead of time and will be provided to the model for both the historical and forecast window.

In [57]:
means = sales_values.mean(axis=1, keepdims=True)
stds = sales_values.std(axis=1, keepdims=True) + 1e-6

sales_norm = (sales_values - means) / stds

Retail demand varies significantly across products.

We normalize each time series independently:

\[
x_{norm} = \frac{x - \mu}{\sigma}
\]

This stabilizes training and prevents high-volume items from dominating gradients.

In [58]:
for col in ["item_id", "store_id", "cat_id", "state_id"]:
    sales[col] = sales[col].astype("category").cat.codes

In [59]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
static_features = sales[["item_id", "store_id", "cat_id", "state_id"]].values
static_tensor = torch.tensor(static_features).long().to(device)

In [60]:
history = 30
forecast = 7

def create_samples(sales_norm, known_features, history=30, forecast=1):
    """
    Slide a window across all series and collect (series_id, past, known, target) tuples.
    Generates ALL possible windows from ALL series.
    """
    T_total = sales_norm.shape[1]
    samples = []

    for i in range(sales_norm.shape[0]):
        for t in range(history, T_total - forecast):
            past   = sales_norm[i, t - history : t]
            target = sales_norm[i, t : t + forecast]
            known  = known_features[t - history : t + forecast]
            samples.append((i, past, known, target))

    return samples

samples = create_samples(sales_norm, known_features)
print(f"Total samples: {len(samples)}")


Total samples: 73800


In [61]:
def split_samples(samples, train_ratio=0.8):
    """
    Split samples into train/val by index (temporal order is preserved).
    """
    split_idx = int(len(samples) * train_ratio)
    return samples[:split_idx], samples[split_idx:]

train_samples, val_samples = split_samples(samples)


In [62]:
def build_tensors(samples):
    series_ids = torch.tensor([s[0] for s in samples])
    past_tensor = torch.tensor([s[1] for s in samples]).float().unsqueeze(-1)
    known_tensor = torch.tensor([s[2] for s in samples]).float()
    target_tensor = torch.tensor([s[3] for s in samples]).float()

    return series_ids, past_tensor, known_tensor, target_tensor

train_ids, train_past, train_known, train_target = build_tensors(train_samples)
val_ids, val_past, val_known, val_target = build_tensors(val_samples)

We convert the sliding window samples into PyTorch tensors:

- `past_tensor` → [N, 30, 1]
- `known_tensor` → [N, 37, K]
- `target_tensor` → [N, 7]
- `series_ids` → used to fetch static features

This prepares the dataset for model training.

In [63]:
train_ids = train_ids.to(device)
train_past = train_past.to(device)
train_known = train_known.to(device)
train_target = train_target.to(device)

val_ids = val_ids.to(device)
val_past = val_past.to(device)
val_known = val_known.to(device)
val_target = val_target.to(device)

In [64]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------
# MODEL
# -----------------------
model = MinimalTFT(
    num_items=num_items,
    num_stores=num_stores,
    num_cats=num_cats,
    num_states=num_states,
    past_dim=1,
    future_dim=5,
    forecast_len=1,
    d_model=128,
    num_heads=8,
).to(device)

# -----------------------
# OPTIMIZER + LOSS
# -----------------------
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
criterion = nn.L1Loss()

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    patience=3,
    factor=0.5
)

batch_size = 64
num_epochs = 30
patience = 6
best_val_loss = float("inf")
early_stop_counter = 0

# -----------------------
# DEBUG: Check target scale
# -----------------------
print("Target min:", train_target.min().item())
print("Target max:", train_target.max().item())

# -----------------------
# TRAINING LOOP
# -----------------------
for epoch in range(num_epochs):

    model.train()
    perm = torch.randperm(len(train_past))

    total_train_loss = 0
    num_batches = 0

    for i in range(0, len(train_past), batch_size):

        idx = perm[i:i+batch_size]

        static = static_tensor[train_ids[idx]].to(device)
        past = train_past[idx].to(device)
        known = train_known[idx].to(device)
        target = train_target[idx].to(device)

        optimizer.zero_grad()

        output = model(static, past, known)

        # Debug shape check (only first epoch first batch)
        if epoch == 0 and i == 0:
            print("Output shape:", output.shape)
            print("Target shape:", target.shape)

        loss = criterion(output, target)

        loss.backward()

        # Gradient clipping (important for transformers)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        total_train_loss += loss.item()
        num_batches += 1

    avg_train_loss = total_train_loss / num_batches

    # -----------------------
    # VALIDATION
    # -----------------------
    model.eval()
    with torch.no_grad():

        static_val = static_tensor[val_ids].to(device)
        val_output = model(static_val, val_past.to(device), val_known.to(device))
        val_loss = criterion(val_output, val_target.to(device))

    scheduler.step(val_loss)

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {val_loss.item():.4f} | "
        f"LR: {optimizer.param_groups[0]['lr']:.6f}"
    )

    # -----------------------
    # EARLY STOPPING
    # -----------------------
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        early_stop_counter = 0
        torch.save(model.state_dict(), "best_model.pt")
    else:
        early_stop_counter += 1

    if early_stop_counter >= patience:
        print("Early stopping triggered.")
        break


# -----------------------
# NAIVE BASELINE CHECK
# -----------------------
print("\n--- Naive Baseline Check ---")

with torch.no_grad():
    naive_pred = val_past[:, -1:, :].repeat(1, 1, 1).to(device)
    naive_mae = torch.mean(torch.abs(naive_pred - val_target.to(device)))

print("Naive MAE:", naive_mae.item())
print("Best Val MAE:", best_val_loss.item())

Target min: -1.782309651374817
Target max: 17.861536026000977
Output shape: torch.Size([64, 1])
Target shape: torch.Size([64, 1])
Epoch 00 | Train Loss: 0.6058 | Val Loss: 0.5864 | LR: 0.000300
Epoch 01 | Train Loss: 0.5933 | Val Loss: 0.5835 | LR: 0.000300
Epoch 02 | Train Loss: 0.5908 | Val Loss: 0.5756 | LR: 0.000300
Epoch 03 | Train Loss: 0.5890 | Val Loss: 0.5732 | LR: 0.000300
Epoch 04 | Train Loss: 0.5879 | Val Loss: 0.5781 | LR: 0.000300
Epoch 05 | Train Loss: 0.5869 | Val Loss: 0.5734 | LR: 0.000300
Epoch 06 | Train Loss: 0.5852 | Val Loss: 0.5730 | LR: 0.000300
Epoch 07 | Train Loss: 0.5839 | Val Loss: 0.5798 | LR: 0.000300
Epoch 08 | Train Loss: 0.5831 | Val Loss: 0.5730 | LR: 0.000300
Epoch 09 | Train Loss: 0.5815 | Val Loss: 0.5759 | LR: 0.000300
Epoch 10 | Train Loss: 0.5836 | Val Loss: 0.5740 | LR: 0.000150
Epoch 11 | Train Loss: 0.5794 | Val Loss: 0.5732 | LR: 0.000150
Epoch 12 | Train Loss: 0.5784 | Val Loss: 0.5811 | LR: 0.000150
Early stopping triggered.

--- Naive B